In [1]:
import os
import random
import torch
import re
import string

import numpy as np
import pandas as pd

from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel

from nltk.corpus import stopwords


from interpreto.concepts import ICAConcepts
from interpreto import ModelWithSplitPoints
from interpreto.concepts.interpretations import TopKInputs
from interpreto import plot_concepts

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# Set up constant variables.
INPUT_FOLDER = 'data'
OUTPUT_FOLDER = 'explanations'
DEVICE = 0 if torch.cuda.is_available() else -1

# Set up stop words.
STOP_WORDS = set(stopwords.words("english"))
clinical_keep = {} # Clinical words from stopwords we wish to keep
clinical_add = ['pm', 'patient', 'resident'] # words common in nurse notes we also wish to remove
FILTERED_WORDS = [w for w in STOP_WORDS if w not in clinical_keep]
FILTERED_WORDS.extend(clinical_add) 
FILTERED_WORDS.extend([char for char in string.punctuation]) # extend filtered words to include punctuation

# Make model variables.
MODEL_NAME = "nlpie/tiny-clinicalbert"
PEFT_HEAD = "synth_lora_model_tinyclinicalbert"


dico_name_classes = {
    0: "met",
    1: "unmet"
}

In [3]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state
RANDOM_STATE = set_random_states(1618)

In [ ]:
# Get real data to work with.
all_data = {}
for folder in os.listdir(f'../../../{INPUT_FOLDER}/'):
    if '.' not in folder:
        for sub_folder in os.listdir(f'../../../{INPUT_FOLDER}/{folder}'):
            if '.' not in sub_folder and 'T1' in sub_folder:
                for sub_sub_folder in os.listdir(f'../../../{INPUT_FOLDER}/{folder}/{sub_folder}'):
                    if '.' not in sub_sub_folder: 
                        daily_nurse = pd.read_excel(f'../../../{INPUT_FOLDER}/{folder}/{sub_folder}/{sub_sub_folder}/dailyNurseNotes_{sub_sub_folder.split(' ')[0]}.xlsx')
                        daily_nurse = daily_nurse.dropna(subset=['Note', 'Date', 'Time'])
                        all_data[f"{sub_sub_folder.split(' ')[0]}"] = daily_nurse['Note'].dropna().values.tolist()

In [5]:
# Load model.
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model_classif = PeftModel.from_pretrained(base_model, PEFT_HEAD)
tokenizer_classif = AutoTokenizer.from_pretrained(PEFT_HEAD)
merged_model = model_classif.merge_and_unload()

Loading weights:   0%|          | 0/69 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nlpie/tiny-clinicalbert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expe

In [6]:
model_with_split_points = ModelWithSplitPoints(
    model_or_repo_id=merged_model,
    tokenizer=tokenizer_classif,
    split_points=[2],  # split at the sixth layer
    device_map="cuda",
    batch_size=8,
)

In [7]:
# Prepare dataset.
inputs = []

for patient_id in all_data:
    inputs.extend(all_data[patient_id])

# Clean inputs to remove noisy tokens.
def clean_inputs(inputs):
    cleaned = []
    
    for note in inputs:
        words = note.split()
        
        words = [
            w.lower()
            for w in words
            if re.match(r"^[a-zA-Z]{2,}$", w)   # only alphabetic words ≥2 chars
        ]
        
        cleaned.append(" ".join(words))
        
    return cleaned

inputs = clean_inputs(inputs)

# limit size for faster testing (this is optional)
# inputs = inputs[:1000]

print(f"Number of notes: {len(inputs)}")

# Compute CLS activations
granularity = ModelWithSplitPoints.activation_granularities.CLS_TOKEN

activations = model_with_split_points.get_activations(
    inputs=inputs,
    activation_granularity=granularity,
    include_predicted_classes=False
)

Number of notes: 12373


In [8]:
# Make the concept explainer. (Why ICA here: https://for-sight-ai.github.io/interpreto/notebooks/classification_concept_tutorial/#:~:text=ICAConcepts%20is%20a%20good%20first%20candidate%20for%20classification%2E%20It%20has%20no%20requirements%2C%20is%20fast%2C%20and%20provide%20correct%20first%20results%20on%20most%20datasets%2E)
concept_explainer = ICAConcepts(
    model_with_split_points, 
    nb_concepts=50, 
    device="cuda")
# Fit the concept explainer on activations.
concept_explainer.fit(activations)

In [9]:
# instantiate the interpretation method with the concept explainer
topk_inputs_method = TopKInputs(
    concept_explainer=concept_explainer,
    k=5,
    activation_granularity=granularity,
    use_unique_words=True,  # with the [CLS] token granularity, we are forced to use unique words
    unique_words_kwargs={
        "count_min_threshold": round(
            len(inputs) * 0.002
        ),  # appear in at least 0.2% of the samples | increase if random words appear and decrease if some words appear too often
        "lemmatize": True,
        "words_to_ignore": FILTERED_WORDS,  # include noise words and punctuation
    },
)

In [10]:
# call the interpretation methods on the inputs
# we cannot give the previously computed activations because `use_unique_words=True` creates samples with a single word inside
topk_words = topk_inputs_method.interpret(
    inputs=inputs,
    concepts_indices="all",
)

In [11]:
# estimate the importance of concepts for each class using the gradient
gradients = concept_explainer.concept_output_gradient(
    inputs=inputs,
    targets=None,  # None means all classes
    activation_granularity=granularity,
    concepts_x_gradients=True,  # the concept to output gradients are multiplied by the concepts values, this is common practice in the literature
    batch_size=64,
)

# stack gradients on samples and average them over samples
mean_gradients = torch.stack(gradients).abs().squeeze().mean(0)  # (num_classes, num_concepts)

# for each class, sort the importance scores
order = torch.argsort(mean_gradients, descending=True)

# visualize the top 5 concepts for each class
for target in range(order.shape[0]):
    for i in range(5):
        concept_id = order[target, i].item()
        importance = mean_gradients[target, concept_id].item()
        words = list(topk_words.get(concept_id, None).keys())
        print(f"\tconcept id: {concept_id},\timportance: {round(importance, 3)},\ttopk words: {words}")

	concept id: 35,	importance: 0.051,	topk words: ['px', 'tds', 'hr', 'note', 'dose']
	concept id: 22,	importance: 0.045,	topk words: ['small', 'explained', 'fair', 'reassured', 'noticed']
	concept id: 7,	importance: 0.041,	topk words: ['ging', 'sign', 'dressing', 'toileting', 'abx']
	concept id: 39,	importance: 0.039,	topk words: ['vomiting', 'altered', 'vaccination', 'mobilising', 'fall']
	concept id: 10,	importance: 0.038,	topk words: ['pleasantly', 'reassurance', 'assessment', 'prn', 'spent']
	concept id: 22,	importance: 0.049,	topk words: ['small', 'explained', 'fair', 'reassured', 'noticed']
	concept id: 35,	importance: 0.049,	topk words: ['px', 'tds', 'hr', 'note', 'dose']
	concept id: 39,	importance: 0.042,	topk words: ['vomiting', 'altered', 'vaccination', 'mobilising', 'fall']
	concept id: 47,	importance: 0.039,	topk words: ['right', 'staff', 'one', 'yesterday', 'mid']
	concept id: 36,	importance: 0.039,	topk words: ['nurse', 'spent', 'slept', 'encouraged', 'hour']


In [12]:
labels = {k: list(v.keys()) for k, v in topk_words.items()}

plot_concepts(
    # classes_names=classes_names,
    concepts_importances=mean_gradients,
    concepts_labels=labels,
)

In [13]:
labels = {k: list(v.keys()) for k, v in topk_words.items()}

plot_concepts(
    # classes_names=classes_names,
    concepts_importances=mean_gradients,
    concepts_labels=labels,
)